**RULES FOR A GOOD TARGET**

In [1]:
import numpy as np
import pandas as pd

df= pd.read_csv("Datasets/clean_final_data.csv")

def is_potential_target(df,col):

    # Rule 1: Must be categorical(not continous number)
    if df[col].dtype in ['int64','float64']:
        # Exception: if it has few unique values, could be categorical
        n_unique = df[col].nunique()
        if n_unique > 10:
            return False, f"Too many unique values ({n_unique})"
        

    # Rule 2: Should have reasonable number of classes
    n_unique = df[col].nunique()
    if n_unique < 2:
        return False, "Only 1 unique value"
    
    if n_unique > 20:
        return False,f"Too many classes ({n_unique})"
    
    # Rule 3 : Not an identifier
    id_keywords = ['id','ID','uuid','key','code']
    if any(kw in col for kw in id_keywords):
        return False, 'Looks like an identifier'

    #Rule 4 : Not a date"
    if "date" in col.lower() or 'time' in col.lower():
        return False, 'Date/time columns'
    
    # Rule 5 Not a duplicate/calculate field
    #(this is the harder to detect automatically)


    return True, f"Potential target ({n_unique} classes)"

# Check All columns
print('='*70)
print("IDENTIFY POTENTIAL TARGETS")
print("="*70)

potential_targets = []
for col in df.columns:
    is_target,reason = is_potential_target(df,col)
    symbol = '✅' if is_target else "❌"
    print(f"{symbol} {col:20s} | {reason}")
    if is_target:
        potential_targets.append(col)

print(f"\n Potential targets: {potential_targets}")


IDENTIFY POTENTIAL TARGETS
❌ OrderID              | Too many unique values (49222)
❌ CustomerID           | Too many unique values (9820)
❌ OrderDate            | Too many classes (912)
❌ ProductID            | Too many unique values (20)
✅ Quantity             | Potential target (8 classes)
✅ Discount             | Potential target (6 classes)
✅ PaymentMethod        | Potential target (4 classes)
✅ Status               | Potential target (3 classes)
❌ Age                  | Too many unique values (48)
✅ City                 | Potential target (10 classes)
❌ SignupDate           | Too many classes (1094)
✅ CustomerSegment      | Potential target (3 classes)
✅ ProductName          | Potential target (20 classes)
✅ Category             | Potential target (6 classes)
❌ UnitPrice            | Too many unique values (20)
❌ Sales                | Too many unique values (290)
❌ OrderValue           | Too many unique values (290)

 Potential targets: ['Quantity', 'Discount', 'PaymentMethod', '

**ANALYZING EACH POTENTIAL TARGET**

In [2]:
# Run this to increase output limit
from IPython.display import display, HTML
display(HTML("<style>.output_area { max-height: 500px !important; overflow-y: auto !important; }</style>"))
def analyze_target(df,target):


    print(f"\n{'='*70}")
    print(f"TARGET ANALYSIS: {target}")
    print(f"{'='*70}")

    #Basic stats 
    n_classes = df[target].unique()
    print(f"\nNumber of classes : {n_classes}")

    # Class distribution
    counts = df[target].value_counts()
    percentages = df[target].value_counts(normalize = True) * 100

    print(f"\nClass distribution:")
    for cls in counts.index:
        print(f" {cls} : {counts[cls]} ({percentages[cls]:.2f}%)")

    # Balance check
    min_pct = percentages.min()
    max_pct = percentages.max()

    if min_pct < 1:
        balance = "❌ SERVERLY BALANCED"  
    elif min_pct < 5 :
         balance ="⚠️ IMBALANCED"
    elif min_pct <15:
        balance="⚠️ MODERATELY IMBALANCED"
    else:
        balance = "✅ BALANCED"

    print(f"\nBalance: {balance}")
    print(f"Min class: {min_pct:.2f}%")
    print(f"Max class: {max_pct:.2f}%")

    # Business interpretation
    print(f"\nBusiness question: would predicting '{target}' be useful")
    print(f"- Who would use this prediction?")
    print(f" -What action would they take ?")
    print(f" -Does it matter if were wrong?")

    return {
        'classes': n_classes,
        'balance':balance,
        'distribution': counts.to_dict()
    }

# Analyze your shortlist
for target in ['PaymentMethod','Status','CustomerSegment','Category']:
    analyze_target(df, target)



TARGET ANALYSIS: PaymentMethod

Number of classes : <StringArray>
['Gateway', 'Wallet', 'CardToCard', 'Cash']
Length: 4, dtype: str

Class distribution:
 Gateway : 23894 (48.54%)
 CardToCard : 11744 (23.86%)
 Wallet : 8661 (17.60%)
 Cash : 4923 (10.00%)

Balance: ⚠️ MODERATELY IMBALANCED
Min class: 10.00%
Max class: 48.54%

Business question: would predicting 'PaymentMethod' be useful
- Who would use this prediction?
 -What action would they take ?
 -Does it matter if were wrong?

TARGET ANALYSIS: Status

Number of classes : <StringArray>
['Completed', 'Returned', 'Cancelled']
Length: 3, dtype: str

Class distribution:
 Completed : 45276 (91.98%)
 Cancelled : 2438 (4.95%)
 Returned : 1508 (3.06%)

Balance: ⚠️ IMBALANCED
Min class: 3.06%
Max class: 91.98%

Business question: would predicting 'Status' be useful
- Who would use this prediction?
 -What action would they take ?
 -Does it matter if were wrong?

TARGET ANALYSIS: CustomerSegment

Number of classes : <StringArray>
['Regular', 

In [3]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['SignupDate'] = pd.to_datetime(df['SignupDate'])
df['CustomerTenure'] = (df['OrderDate'] - df['SignupDate']).dt.days

customer_Stats = df.groupby('CustomerID').agg({
    'OrderID' :'count',
    'Sales':'sum',
}).reset_index()
customer_Stats.columns = ['CustomerID','OrderCount','TotalSpend']

# Merge back to add new columns

df = df.merge(customer_Stats, on = 'CustomerID',how = 'left')

# Create interaction columns

df['RevenuePerItem'] = df['Sales']/ df['Quantity']
df['IsDiscounted'] = (df['Discount'] > 0).astype(int)

# Now check your columns

print('\nYour New Columns')
print(df.columns.tolist())





Your New Columns
['OrderID', 'CustomerID', 'OrderDate', 'ProductID', 'Quantity', 'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'SignupDate', 'CustomerSegment', 'ProductName', 'Category', 'UnitPrice', 'Sales', 'OrderValue', 'CustomerTenure', 'OrderCount', 'TotalSpend', 'RevenuePerItem', 'IsDiscounted']


**F1 TEST**

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

def quick_target_test(df,target,features):

    """
    Quick test if a target is learnable from iven features.
    Returns the F1 macro score
    """
    X = df[features].copy()
    y=df[target]

    # Encode Categorica; features
    for col in X.select_dtypes(include=['str','object']).columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

    # Handle NaN
    X = X.fillna(X.median(numeric_only=True))

    # Encode target
    y_encoded = LabelEncoder().fit_transform(y)

    # Test with RandomForest
    rf= RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)
    scores = cross_val_score(rf,X,y_encoded,cv = 3,scoring='f1_macro',n_jobs=-1)

    return scores.mean()


features = ['Quantity', 'Discount','Age','City','UnitPrice','CustomerTenure','OrderCount','TotalSpend','RevenuePerItem','IsDiscounted']

# Test only the CANDIDATES
candidates = ['PaymentMethod','Status','CustomerSegment','Category']

for target in candidates:
    # Verify no leakage

    if target in features:
        print(f" Skipping {target} - it's in feature!")
        continue

    # Run F1 test
    score = quick_target_test(df,target,features)
    print(f"{target}: F1 = {score:.4f}")
    

PaymentMethod: F1 = 0.1984
Status: F1 = 0.3198
CustomerSegment: F1 = 0.4930
Category: F1 = 1.0000


In [5]:
df.columns

Index(['OrderID', 'CustomerID', 'OrderDate', 'ProductID', 'Quantity',
       'Discount', 'PaymentMethod', 'Status', 'Age', 'City', 'SignupDate',
       'CustomerSegment', 'ProductName', 'Category', 'UnitPrice', 'Sales',
       'OrderValue', 'CustomerTenure', 'OrderCount', 'TotalSpend',
       'RevenuePerItem', 'IsDiscounted'],
      dtype='str')